# 8. TokenisierungAus *Transformer von Grund auf*.

In [ ]:
# In Colab liegt nichts auf der Festplatte, also holt diese Zelle# die zwei oder drei Dateien, die das Kapitel braucht. Wer das# Repository geklont hat, findet sie schon vor, und es wird# nichts heruntergeladen.import osimport sysimport urllib.requestBASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/notebooks/de/code/"PLACES = [".", "code", "../code"]for name in ["ch08_bpe.py"]:    if not any(os.path.exists(os.path.join(where, name))               for where in PLACES):        urllib.request.urlretrieve(BASE + name, name)for where in PLACES:    if os.path.isdir(where) and where not in sys.path:        sys.path.insert(0, where)

Byte-Pair-Encoding, vor den Augen des Lesers trainiert.Beginne mit einzelnen Zeichen. Finde immer wieder das häufigste benachbarte Paar und verschmilz es zu einem neuen Symbol. Das ist der ganze Algorithmus, und er erzeugt das Vokabular fast jedes Modells.

In [ ]:
from collections import CounterCORPUS = (    "die katze trank die milch und die katze schlief "    "die maus trank die sahne und die maus schlief "    "eine katze trank und eine maus schlief "    "die milch und die sahne und die milch wieder")

Jedes Wort wird ein Tupel aus Zeichen, mit Wortende-Marker.

In [ ]:
def word_counts(text):    counts = Counter(text.split())    return {tuple(word) + ("_",): n for word, n in counts.items()}

Wie oft jedes benachbarte Paar im ganzen Korpus vorkommt.

In [ ]:
def pair_counts(words):    pairs = Counter()    for symbols, count in words.items():        for a, b in zip(symbols, symbols[1:]):            pairs[(a, b)] += count    return pairs

Jedes Vorkommen eines Paares durch ein Symbol ersetzen.

In [ ]:
def merge(words, pair):    joined = pair[0] + pair[1]    out = {}    for symbols, count in words.items():        new, i = [], 0        while i < len(symbols):            if (i < len(symbols) - 1                    and (symbols[i], symbols[i + 1]) == pair):                new.append(joined)                i += 2            else:                new.append(symbols[i])                i += 1        out[tuple(new)] = count    return out

Der Algorithmus, mit allen Verschmelzungen und Zwischenständen.

In [ ]:
def train(text, merges):    words = word_counts(text)    history, learned = [], []    for step in range(merges):        pairs = pair_counts(words)        if not pairs:            break        pair, count = pairs.most_common(1)[0]        words = merge(words, pair)        learned.append(pair)        history.append((step + 1, pair, count, tokens_in(words)))    return learned, words, history

In [ ]:
def tokens_in(words):    return sum(len(symbols) * count for symbols, count in words.items())

Die Verschmelzungen in gelernter Reihenfolge anwenden.

In [ ]:
def encode(word, learned):    symbols = tuple(word) + ("_",)    for pair in learned:        symbols = tuple(merge({symbols: 1}, pair))[0]    return symbols

In [ ]:
def vocabulary(learned, text):    letters = {c for c in text if c != " "} | {"_"}    return sorted(letters | {a + b for a, b in learned})

In [ ]:
def main():    print("DAS KORPUS, ALS ZEICHEN")    words = word_counts(CORPUS)    print(f"  {len(words)} verschiedene Wörter, {tokens_in(words)} "          f"Tokens vor jeder Verschmelzung")    learned, words, history = train(CORPUS, 12)    print("\nZWÖLF VERSCHMELZUNGEN, HÄUFIGSTES PAAR ZUERST")    print("  Nr.   neu        Anz.  Tokens übrig")    for step, pair, count, total in history:        joined = (pair[0] + pair[1]).replace("_", "␣")        print(f"  {step:<5} {joined:<10} {count:<5} {total}")    print("\nWAS DAS VOKABULAR JETZT ENTHÄLT")    vocab = vocabulary(learned, CORPUS)    print(f"  {len(vocab)} Symbole")    line = "  "    for symbol in (v.replace("_", "␣") for v in vocab):        if len(line) + len(symbol) + 1 > 70:            print(line)            line = "  "        line += " " + symbol    print(line)    print("\nWÖRTER MIT DIESEN VERSCHMELZUNGEN KODIEREN")    for word in ["die", "katze", "maus", "trank", "elefant"]:        pieces = encode(word, learned)        shown = " ".join(p.replace("_", "␣") for p in pieces)        count = f"{len(pieces)} Teil" + ("" if len(pieces) == 1 else "e")        print(f"  {word:<10} {count + ':':<10} {shown}")    print("\nMEHR VERSCHMELZUNGEN, WENIGER TEILE PRO WORT")    print("  Verschm. Vokabular  Tokens für das Korpus")    for n in [0, 5, 10, 20, 40, 80]:        got, ended, _ = train(CORPUS, n)        print(f"  {n:<8} {len(vocabulary(got, CORPUS)):<11} "              f"{tokens_in(ended)}")

In [ ]:
main()

Das Kapitel, aus dem dieses Notebook stammt, rechnet jede Zahl oben durch und erklärt, warum jeder Schritt da ist.